# Plan Your Trip with Kayak

## Contexte du projet

**Kayak**, moteur de recherche de voyages (Booking Holdings), souhaite une application de recommandation de destinations basée sur **météo** et **hôtels** pour **35 villes** de France.

### Objectifs (cahier des charges)

1. Coordonnées GPS (Nominatim) pour chaque ville
2. Prévisions météo 7 jours (OpenWeatherMap One Call 3.0)
3. Scraping Booking.com pour **chaque destination** (liste des 35 villes)
4. **Data Lake** : fichiers CSV sur **Amazon S3** sous `Bloc-1/Kayak/raw/`
5. **ETL** : chargement nettoyé vers **PostgreSQL managé (Neon)** — data warehouse (variable `NEON_DATABASE_URL`)
6. Cartes **Top-5** destinations et **Top-20** hôtels (Plotly)

### Implémentation retenue

- **Neon** comme base PostgreSQL managée pour l’entrepôt de données.
- **S3** : bucket lab et préfixe **`cloudlab-certification-b6bff020/Bloc-1/Kayak/`** pour les CSV.
- **Hôtels** : une recherche par ville sur l’ensemble des **35 destinations** (pas limitée au classement météo).
- **CSV enrichi** `enriched_kayak.csv` : jointure hôtels et indicateurs météo agrégés par ville.


## Schéma d'infrastructure

```
SOURCES          →    Python (collecte)    →    S3 (Data Lake)    →    ETL    →    Neon (DWH)    →    Plotly
Nominatim             pandas / requests         Bloc-1/Kayak/raw/         pandas       PostgreSQL
OpenWeatherMap        Selenium / BS4            CSV
Booking.com
```

| Composant | Rôle |
|-----------|------|
| S3 | Stockage des CSV bruts (zone `raw/`) |
| Neon | Entrepôt relationnel, requêtes SQL pour les équipes analytics |
| IAM | Droits **S3 uniquement** pour ce projet |


---
## 1. Configuration et imports


In [1]:
!pip install requests pandas numpy plotly boto3 sqlalchemy psycopg2-binary python-dotenv beautifulsoup4 selenium webdriver-manager lxml -q


In [2]:
import os
import re
import time
import json
import warnings
from datetime import datetime, timedelta
from io import StringIO

import requests
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)


In [3]:
load_dotenv()

OPENWEATHERMAP_API_KEY = os.getenv("OPENWEATHERMAP_API_KEY")
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "eu-west-3")
S3_BUCKET_NAME = os.getenv("S3_BUCKET_NAME", "cloudlab-certification-b6bff020")
S3_PREFIX = os.getenv("S3_PREFIX", "Bloc-1/Kayak").strip().strip("/")
NEON_DATABASE_URL = os.getenv("NEON_DATABASE_URL")

assert OPENWEATHERMAP_API_KEY, "OPENWEATHERMAP_API_KEY manquante dans .env"
print("Configuration chargée (S3 :", S3_BUCKET_NAME, "| préfixe :", S3_PREFIX + ")")


Configuration chargée (S3 : cloudlab-certification-b6bff020 | préfixe : Bloc-1/Kayak)


In [4]:
cities = [
    "Mont Saint Michel", "St Malo", "Bayeux", "Le Havre", "Rouen",
    "Paris", "Amiens", "Lille", "Strasbourg", "Chateau du Haut Koenigsbourg",
    "Colmar", "Eguisheim", "Besancon", "Dijon", "Annecy",
    "Grenoble", "Lyon", "Gorges du Verdon", "Bormes les Mimosas", "Cassis",
    "Marseille", "Aix en Provence", "Avignon", "Uzes", "Nimes",
    "Aigues Mortes", "Saintes Maries de la mer", "Collioure", "Carcassonne", "Ariege",
    "Toulouse", "Montauban", "Biarritz", "Bayonne", "La Rochelle",
]

os.makedirs("src", exist_ok=True)
print(f"{len(cities)} villes à traiter.")


35 villes à traiter.


---
## 2. Collecte des coordonnées GPS (API Nominatim)

[Nominatim](https://nominatim.org/) — politique : **1 requête / seconde**, `User-Agent` identifié.


In [5]:
def get_city_coordinates(city_name, country_code="fr"):
    """Récupère les coordonnées GPS d'une ville via l'API Nominatim."""
    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": city_name, "countrycodes": country_code,
              "format": "json", "limit": 1}
    headers = {"User-Agent": "KayakTripPlanner/1.0 (student-project)"}
    try:
        resp = requests.get(url, params=params, headers=headers, timeout=10)
        resp.raise_for_status()
        data = resp.json()
        if data:
            return {"city": city_name,
                    "latitude": float(data[0]["lat"]),
                    "longitude": float(data[0]["lon"])}
        print(f"  [!] Aucun résultat pour '{city_name}'")
    except Exception as e:
        print(f"  [!] Erreur pour '{city_name}': {e}")
    return {"city": city_name, "latitude": None, "longitude": None}


print("Collecte des coordonnées GPS via Nominatim…")
print("=" * 55)

city_data = []
for i, city in enumerate(cities, 1):
    print(f"  [{i:2d}/35] {city}…", end=" ")
    result = get_city_coordinates(city)
    result["id"] = i
    city_data.append(result)
    status = f"OK ({result['latitude']:.4f}, {result['longitude']:.4f})" if result["latitude"] else "ÉCHEC"
    print(status)
    time.sleep(1)

cities_df = pd.DataFrame(city_data)[["id", "city", "latitude", "longitude"]]
nb_ok = len(cities_df.dropna(subset=["latitude"]))
print(f"\n{nb_ok}/{len(cities)} villes géolocalisées avec succès.")


Collecte des coordonnées GPS via Nominatim…
OK (48.6360, -1.5115)Michel… 
  [ 2/35] St Malo… OK (48.6495, -2.0260)
  [ 3/35] Bayeux… OK (49.2765, -0.7025)
  [ 4/35] Le Havre… OK (49.4939, 0.1080)
  [ 5/35] Rouen… OK (49.4405, 1.0940)
  [ 6/35] Paris… OK (48.8535, 2.3484)
  [ 7/35] Amiens… OK (49.8942, 2.2957)
  [ 8/35] Lille… OK (50.6366, 3.0635)
  [ 9/35] Strasbourg… OK (48.5846, 7.7507)
  [10/35] Chateau du Haut Koenigsbourg… OK (48.2494, 7.3439)
  [11/35] Colmar… OK (48.0778, 7.3580)
  [12/35] Eguisheim… OK (48.0448, 7.3080)
  [13/35] Besancon… OK (47.2380, 6.0244)
  [14/35] Dijon… OK (47.3216, 5.0415)
  [15/35] Annecy… OK (45.8992, 6.1289)
  [16/35] Grenoble… OK (45.1876, 5.7358)
  [17/35] Lyon… OK (45.7578, 4.8320)
  [18/35] Gorges du Verdon… OK (43.7497, 6.3286)
  [19/35] Bormes les Mimosas… OK (43.1507, 6.3419)
  [20/35] Cassis… OK (43.2140, 5.5396)
  [21/35] Marseille… OK (43.2962, 5.3700)
  [22/35] Aix en Provence… OK (43.5298, 5.4475)
  [23/35] Avignon… OK (43.9492, 4.8059)
 

In [6]:
cities_df


,id,city,latitude,longitude
0,1,Mont Saint Michel,48.635954,-1.511460
1,2,St Malo,48.649518,-2.026041
2,3,Bayeux,49.276462,-0.702474
3,4,Le Havre,49.493898,0.107973
4,5,Rouen,49.440459,1.093966
5,6,Paris,48.853495,2.348391
6,7,Amiens,49.894171,2.295695
7,8,Lille,50.636565,3.063528
8,9,Strasbourg,48.584614,7.750713
9,10,Chateau du Haut Koenigsbourg,48.249382,7.343941


---
## 3. Collecte météo (OpenWeatherMap One Call 3.0)

Les champs `rain` de l'API peuvent être un nombre ou un dictionnaire (`{"1h": x, "3h": y}`) : on les normalise en **millimètres** pour l'agrégation.


In [7]:
def parse_rain_mm(rain_field):
    """Convertit le champ API `rain` en mm numériques."""
    if rain_field is None:
        return 0.0
    if isinstance(rain_field, (int, float)):
        return float(rain_field)
    if isinstance(rain_field, dict):
        return float(sum(float(v) for v in rain_field.values()))
    return 0.0


def get_weather_forecast(lat, lon, api_key):
    """Prévisions 7 jours — OpenWeatherMap One Call 3.0."""
    url = "https://api.openweathermap.org/data/3.0/onecall"
    params = {"lat": lat, "lon": lon, "appid": api_key,
              "units": "metric", "lang": "fr", "exclude": "minutely,hourly,alerts"}
    try:
        resp = requests.get(url, params=params, timeout=15)
        resp.raise_for_status()
        return resp.json()
    except requests.exceptions.HTTPError:
        if resp.status_code == 401:
            print("  [!] Clé API invalide ou quota dépassé.")
        else:
            print(f"  [!] HTTP {resp.status_code}")
        return None
    except Exception as e:
        print(f"  [!] Erreur : {e}")
        return None


print("Collecte des prévisions météo via OpenWeatherMap…")
print("=" * 55)

weather_records = []
for _, row in cities_df.dropna(subset=["latitude"]).iterrows():
    print(f"  [{int(row['id']):2d}/35] {row['city']}…", end=" ")
    data = get_weather_forecast(row["latitude"], row["longitude"], OPENWEATHERMAP_API_KEY)
    if data and "daily" in data:
        for day in data["daily"]:
            weather_records.append({
                "city_id": int(row["id"]), "city": row["city"],
                "latitude": row["latitude"], "longitude": row["longitude"],
                "date": datetime.fromtimestamp(day["dt"]).strftime("%Y-%m-%d"),
                "temp_day": day["temp"]["day"],
                "temp_min": day["temp"]["min"],
                "temp_max": day["temp"]["max"],
                "feels_like": day["feels_like"]["day"],
                "humidity": day["humidity"],
                "wind_speed": day["wind_speed"],
                "pop": day.get("pop", 0),
                "rain_volume": parse_rain_mm(day.get("rain")),
                "weather_main": day["weather"][0]["main"],
                "weather_description": day["weather"][0]["description"],
            })
        print("OK")
    else:
        print("ÉCHEC")
    time.sleep(0.5)

weather_df = pd.DataFrame(weather_records)
print(f"\n{len(weather_df)} enregistrements météo collectés pour {weather_df['city'].nunique()} villes.")


Collecte des prévisions météo via OpenWeatherMap…
OK[ 1/35] Mont Saint Michel… 
  [ 2/35] St Malo… OK
  [ 3/35] Bayeux… OK
  [ 4/35] Le Havre… OK
  [ 5/35] Rouen… OK
  [ 6/35] Paris… OK
  [ 7/35] Amiens… OK
  [ 8/35] Lille… OK
  [ 9/35] Strasbourg… OK
  [10/35] Chateau du Haut Koenigsbourg… OK
  [11/35] Colmar… OK
  [12/35] Eguisheim… OK
  [13/35] Besancon… OK
  [14/35] Dijon… OK
  [15/35] Annecy… OK
  [16/35] Grenoble… OK
  [17/35] Lyon… OK
  [18/35] Gorges du Verdon… OK
  [19/35] Bormes les Mimosas… OK
  [20/35] Cassis… OK
  [21/35] Marseille… OK
  [22/35] Aix en Provence… OK
  [23/35] Avignon… OK
  [24/35] Uzes… OK
  [25/35] Nimes… OK
  [26/35] Aigues Mortes… OK
  [27/35] Saintes Maries de la mer… OK
  [28/35] Collioure… OK
  [29/35] Carcassonne… OK
  [30/35] Ariege… OK
  [31/35] Toulouse… OK
  [32/35] Montauban… OK
  [33/35] Biarritz… OK
  [34/35] Bayonne… OK
  [35/35] La Rochelle… OK

280 enregistrements météo collectés pour 35 villes.


In [8]:
weather_df.head(10)


,city_id,city,latitude,longitude,date,temp_day,temp_min,temp_max,feels_like,humidity,wind_speed,pop,rain_volume,weather_main,weather_description
0,1,Mont Saint Michel,48.635954,-1.511460,2026-03-24,12.84,7.12,14.42,11.85,64,8.61,0.00,0.00,Clouds,couvert
1,1,Mont Saint Michel,48.635954,-1.511460,2026-03-25,8.51,6.22,9.98,3.69,61,13.27,1.00,4.42,Rain,légère pluie
2,1,Mont Saint Michel,48.635954,-1.511460,2026-03-26,9.44,4.96,9.49,6.66,55,8.96,0.98,0.93,Rain,légère pluie
3,1,Mont Saint Michel,48.635954,-1.511460,2026-03-27,9.36,5.28,10.90,7.00,82,4.48,0.00,0.00,Clouds,couvert
4,1,Mont Saint Michel,48.635954,-1.511460,2026-03-28,9.47,5.49,9.97,6.91,65,5.99,1.00,1.74,Rain,légère pluie
5,1,Mont Saint Michel,48.635954,-1.511460,2026-03-29,9.65,4.02,9.65,6.60,58,7.30,0.27,0.00,Clouds,partiellement nuageux
6,1,Mont Saint Michel,48.635954,-1.511460,2026-03-30,12.62,8.14,12.81,12.05,81,5.88,0.00,0.00,Clouds,couvert
7,1,Mont Saint Michel,48.635954,-1.511460,2026-03-31,15.45,8.78,15.79,14.83,68,5.16,0.00,0.00,Clouds,nuageux
8,2,St Malo,48.649518,-2.026041,2026-03-24,12.64,7.96,14.30,11.71,67,10.39,0.00,0.00,Clouds,nuageux
9,2,St Malo,48.649518,-2.026041,2026-03-25,8.61,7.45,10.14,3.70,57,15.04,1.00,3.69,Rain,légère pluie


In [9]:
def compute_weather_score(row):
    """
    Score météo composite (0-100). Pondération : température (40 %), pluie (30 %),
    humidité (15 %), vent (15 %).
    """
    t = row["temp_day"]
    if 20 <= t <= 28:
        temp_s = 100
    elif 15 <= t < 20 or 28 < t <= 33:
        temp_s = 70
    elif 10 <= t < 15 or 33 < t <= 38:
        temp_s = 40
    else:
        temp_s = 10

    rain_s = max(0, 100 * (1 - row["pop"]))
    h = row["humidity"]
    if 30 <= h <= 60:
        hum_s = 100
    elif 20 <= h < 30 or 60 < h <= 75:
        hum_s = 60
    else:
        hum_s = 20
    wind_s = max(0, 100 - row["wind_speed"] * 5)
    return round(0.40 * temp_s + 0.30 * rain_s + 0.15 * hum_s + 0.15 * wind_s, 1)


weather_df["weather_score"] = weather_df.apply(compute_weather_score, axis=1)

city_scores = (
    weather_df
    .groupby(["city_id", "city", "latitude", "longitude"])
    .agg(avg_score=("weather_score", "mean"),
         avg_temp=("temp_day", "mean"),
         max_temp=("temp_max", "max"),
         min_temp=("temp_min", "min"),
         avg_humidity=("humidity", "mean"),
         total_rain=("rain_volume", "sum"),
         avg_pop=("pop", "mean"),
         avg_wind=("wind_speed", "mean"))
    .reset_index()
    .sort_values("avg_score", ascending=False)
    .reset_index(drop=True)
)
city_scores["rank"] = range(1, len(city_scores) + 1)
city_scores = city_scores.round(2)

weather_df.to_csv("src/weather_data.csv", index=False, encoding="utf-8")
city_scores.to_csv("src/city_weather_scores.csv", index=False, encoding="utf-8")
print("Sauvegardé : src/weather_data.csv, src/city_weather_scores.csv")

city_scores[["rank", "city", "avg_score", "avg_temp", "avg_humidity", "total_rain"]].head(10)


Sauvegardé : src/weather_data.csv, src/city_weather_scores.csv


,rank,city,avg_score,avg_temp,avg_humidity,total_rain
0,1,Bormes les Mimosas,74.61,14.42,45.38,0.0
1,2,Aix en Provence,74.27,14.82,40.00,0.0
2,3,Marseille,74.09,14.36,47.62,0.0
3,4,Aigues Mortes,73.79,14.92,45.75,0.0
4,5,Collioure,73.31,15.50,47.62,0.0
5,6,Nimes,72.32,14.47,44.25,0.0
6,7,Cassis,69.42,13.76,48.00,0.0
7,8,Gorges du Verdon,68.74,11.90,34.50,0.0
8,9,Uzes,67.88,13.61,49.88,0.0
9,10,Avignon,67.62,13.28,53.12,0.0


---
## 4. Analyse météo et Top-5 destinations

Le classement final utilise `city_scores` (moyenne du score journalier sur 7 jours).


In [10]:
top_5 = city_scores.head(5).copy()

print("=" * 60)
print("  TOP 5 DES MEILLEURES DESTINATIONS MÉTÉO")
print("=" * 60)
for _, r in top_5.iterrows():
    print(f"\n  #{int(r['rank'])}  {r['city']}")
    print(f"      Score : {r['avg_score']:.1f}/100")
    print(f"      Temp. moy. : {r['avg_temp']:.1f}°C (min {r['min_temp']:.0f}°C / max {r['max_temp']:.0f}°C)")
    print(f"      Humidité : {r['avg_humidity']:.0f}%  |  Pluie cumulée : {r['total_rain']:.1f} mm")
print("\n" + "=" * 60)


  TOP 5 DES MEILLEURES DESTINATIONS MÉTÉO

  #1  Bormes les Mimosas
      Score : 74.6/100
      Temp. moy. : 14.4°C (min 4°C / max 18°C)
      Humidité : 45%  |  Pluie cumulée : 0.0 mm

  #2  Aix en Provence
      Score : 74.3/100
      Temp. moy. : 14.8°C (min 3°C / max 21°C)
      Humidité : 40%  |  Pluie cumulée : 0.0 mm

  #3  Marseille
      Score : 74.1/100
      Temp. moy. : 14.4°C (min 6°C / max 20°C)
      Humidité : 48%  |  Pluie cumulée : 0.0 mm

  #4  Aigues Mortes
      Score : 73.8/100
      Temp. moy. : 14.9°C (min 7°C / max 18°C)
      Humidité : 46%  |  Pluie cumulée : 0.0 mm

  #5  Collioure
      Score : 73.3/100
      Temp. moy. : 15.5°C (min 6°C / max 20°C)
      Humidité : 48%  |  Pluie cumulée : 0.0 mm



In [11]:
fig_top5 = px.scatter_mapbox(
    top_5, lat="latitude", lon="longitude",
    size="avg_score", color="avg_score",
    hover_name="city",
    hover_data={"avg_score": ":.1f", "avg_temp": ":.1f",
                "avg_humidity": ":.0f", "total_rain": ":.1f",
                "latitude": False, "longitude": False},
    color_continuous_scale="YlOrRd", size_max=30, zoom=5,
    center={"lat": 46.5, "lon": 2.5},
    title="Top 5 des meilleures destinations météo en France",
    labels={"avg_score": "Score météo", "avg_temp": "Temp. moy. (°C)",
            "avg_humidity": "Humidité (%)", "total_rain": "Pluie cumulée (mm)"},
)
fig_top5.update_layout(mapbox_style="open-street-map",
                       margin=dict(r=0, t=50, l=0, b=0), height=600)
fig_top5.show()


---
## 5. Scraping des hôtels (Booking.com)

**Périmètre** : une recherche par **ville** pour chacune des **35 destinations** du cahier des charges.

Paramètres : `MAX_HOTELS_PER_CITY` (défaut 20), pauses entre requêtes pour limiter la charge sur les serveurs.

> Le scraping peut prendre **longtemps** (35 villes × délais). En cas d’échec massif (anti-bot), réessayez plus tard, réduisez temporairement le nombre de villes pour tester, ou utilisez un jeu de données CSV local pour valider la suite du pipeline (S3, ETL, visualisations).


In [12]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup

try:
    from webdriver_manager.chrome import ChromeDriverManager
    DRIVER_PATH = ChromeDriverManager().install()
except Exception:
    DRIVER_PATH = None
    print("[!] webdriver_manager indisponible — Chrome doit être configuré manuellement.")


def create_driver():
    """Navigateur Chrome headless avec en-têtes réalistes."""
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument("--lang=fr-FR")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    if DRIVER_PATH:
        return webdriver.Chrome(service=Service(DRIVER_PATH), options=opts)
    return webdriver.Chrome(options=opts)


In [13]:
def _parse_hotel_card(card):
    """Extrait les champs d'une carte hôtel sur la page de résultats Booking.com."""
    h = {}
    try:
        el = card.select_one("[data-testid='title']")
        h["hotel_name"] = el.get_text(strip=True) if el else None

        el = card.select_one("a[data-testid='title-link']")
        if el and el.get("href"):
            href = el["href"]
            h["url"] = href if href.startswith("http") else "https://www.booking.com" + href
        else:
            h["url"] = None

        el = card.select_one("[data-testid='review-score'] div:first-child")
        if el:
            m = re.search(r"[\d,.]+", el.get_text(strip=True))
            h["user_score"] = float(m.group().replace(",", ".")) if m else None
        else:
            h["user_score"] = None

        el = card.select_one("[data-testid='review-score']")
        if el:
            txt = el.get_text(" ", strip=True)
            m = re.search(r"([\d\s\xa0]+)\s*(?:commentaire|avis|évaluation)", txt, re.I)
            h["nb_reviews"] = int(re.sub(r"\s+", "", m.group(1))) if m else None
        else:
            h["nb_reviews"] = None

        el = (card.select_one("[data-testid='price-and-discounted-price']")
              or card.select_one("span[data-testid='price-and-discounted-price']"))
        if el:
            m = re.search(r"([\d\s\xa0]+)", el.get_text(strip=True).replace(" ", " "))
            h["price_per_night"] = int(re.sub(r"\s+", "", m.group(1))) if m else None
        else:
            h["price_per_night"] = None

        stars = card.select("[data-testid='rating-stars'] span")
        if not stars:
            stars = card.select("div[data-testid='rating-stars'] svg")
        h["stars"] = len(stars) if stars else None

        el = (card.select_one("[data-testid='accommodation-type-name']")
              or card.select_one("[data-testid='recommended-units'] span:first-child"))
        h["property_type"] = el.get_text(strip=True) if el else None

        el = card.select_one("[data-testid='distance']")
        if el:
            m = re.search(r"([\d,\.]+)\s*km", el.get_text(strip=True))
            h["distance_center_km"] = float(m.group(1).replace(",", ".")) if m else None
        else:
            h["distance_center_km"] = None

        el = card.select_one("img[data-testid='image']") or card.select_one("img")
        h["image_url"] = el.get("src") if el else None
    except Exception:
        pass
    return h


def scrape_booking_search(city, checkin, checkout, max_results=20):
    """Scrape la page de résultats Booking.com pour une ville donnée."""
    driver = create_driver()
    hotels = []
    try:
        url = (f"https://www.booking.com/searchresults.fr.html"
               f"?ss={city.replace(' ', '+')}"
               f"&checkin={checkin}&checkout={checkout}"
               f"&group_adults=2&no_rooms=1&selected_currency=EUR")
        print(f"    URL : {url[:90]}…")
        driver.get(url)
        time.sleep(5)

        try:
            driver.find_element(By.CSS_SELECTOR, "button[aria-label*='ermer']").click()
            time.sleep(1)
        except Exception:
            pass

        for _ in range(4):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)

        soup = BeautifulSoup(driver.page_source, "lxml")
        cards = soup.select("[data-testid='property-card']")
        print(f"    {len(cards)} propriétés trouvées.")

        for card in cards[:max_results]:
            h = _parse_hotel_card(card)
            if h.get("hotel_name"):
                hotels.append(h)
    except Exception as e:
        print(f"    [!] Erreur : {e}")
    finally:
        driver.quit()
    return hotels


In [14]:
def scrape_hotel_details(hotel_url, driver=None):
    """Page détail : coordonnées, description, équipements."""
    own = driver is None
    if own:
        driver = create_driver()
    details = {"latitude": None, "longitude": None,
               "description": None, "amenities": None}
    try:
        driver.get(hotel_url)
        time.sleep(4)
        soup = BeautifulSoup(driver.page_source, "lxml")

        el = soup.select_one("a[data-atlas-latlng]")
        if el:
            parts = el["data-atlas-latlng"].split(",")
            if len(parts) == 2:
                details["latitude"] = float(parts[0])
                details["longitude"] = float(parts[1])

        if not details["latitude"]:
            for script in soup.find_all("script", type="application/ld+json"):
                try:
                    ld = json.loads(script.string)
                    geo = ld.get("geo", {})
                    if "latitude" in geo:
                        details["latitude"] = float(geo["latitude"])
                        details["longitude"] = float(geo["longitude"])
                        break
                except Exception:
                    continue

        el = (soup.select_one("[data-testid='property-description']")
              or soup.select_one("#property_description_content"))
        if el:
            details["description"] = el.get_text(" ", strip=True)[:500]

        els = soup.select("[data-testid='property-most-popular-facilities-wrapper'] span")
        if not els:
            els = soup.select(".bui-list__description")
        if els:
            details["amenities"] = ", ".join(e.get_text(strip=True) for e in els[:10])
    except Exception as e:
        print(f"      [!] Erreur détail : {e}")
    finally:
        if own:
            driver.quit()
    return details


In [15]:
# --- Paramètres scraping (cahier des charges : hôtels par destination) ---
MAX_HOTELS_PER_CITY = 20
DELAY_BETWEEN_CITIES_SEC = 5

checkin = (datetime.now() + timedelta(days=7)).strftime("%Y-%m-%d")
checkout = (datetime.now() + timedelta(days=8)).strftime("%Y-%m-%d")

print(f"Dates de recherche : {checkin} → {checkout}")
print("Scraping sur les 35 villes géolocalisées — durée potentiellement longue.")
print("=" * 55)

all_hotels = []
cities_to_scrape = cities_df.dropna(subset=["latitude"]).copy()

for _, row in cities_to_scrape.iterrows():
    city_name = row["city"]
    city_id = int(row["id"])
    print(f"\n[{city_name}] Scraping des hôtels…")

    hotels = scrape_booking_search(city_name, checkin, checkout, max_results=MAX_HOTELS_PER_CITY)
    print(f"  → {len(hotels)} hôtels extraits de la page de recherche.")

    driver = create_driver()
    for idx, h in enumerate(hotels):
        h["city_id"] = city_id
        h["city"] = city_name
        if h.get("url"):
            print(f"  Détails [{idx+1}/{len(hotels)}] {str(h.get('hotel_name', ''))[:40]}…", end=" ")
            det = scrape_hotel_details(h["url"], driver=driver)
            h.update(det)
            print("OK" if det.get("latitude") else "(sans coordonnées)")
            time.sleep(3)
        if not h.get("latitude"):
            h["latitude"] = row["latitude"] + np.random.uniform(-0.01, 0.01)
            h["longitude"] = row["longitude"] + np.random.uniform(-0.01, 0.01)
    driver.quit()

    all_hotels.extend(hotels)
    print(f"  Total {city_name} : {len(hotels)} hôtels.")
    time.sleep(DELAY_BETWEEN_CITIES_SEC)

print(f"\nTotal : {len(all_hotels)} hôtels collectés.")


Dates de recherche : 2026-03-31 → 2026-04-01
Scraping sur les 35 villes géolocalisées — durée potentiellement longue.

[Mont Saint Michel] Scraping des hôtels…
    URL : https://www.booking.com/searchresults.fr.html?ss=Mont+Saint+Michel&checkin=2026-03-31&chec…
    75 propriétés trouvées.
  → 20 hôtels extraits de la page de recherche.
  Détails [1/20] La Vieille Auberge… OK
  Détails [2/20] Hôtel Vert… OK
  Détails [3/20] Auberge Saint Pierre… OK
  Détails [4/20] La Mère Poulard… OK
  Détails [5/20] Le Saint Aubert… OK
  Détails [6/20] Le Mouton Blanc… OK
  Détails [7/20] Mercure Mont Saint Michel… OK
  Détails [8/20] Le Relais Saint Michel… OK
  Détails [9/20] Le Relais Du Roy… OK
  Détails [10/20] Le Marquis De La Guintre… OK
  Détails [11/20] Mon Saint Michel… OK
  Détails [12/20] Apparthôtel Mont Saint Michel - Résidenc… OK
  Détails [13/20] Ermitage - Mont-Saint-Michel… OK
  Détails [14/20] Les Vieilles Digues… OK
  Détails [15/20] Au Mont De La Rive #Jaccuzi et Mont-Sain… OK
  D

In [16]:
HOTEL_COLUMNS = [
    "city_id", "city", "hotel_name", "url",
    "latitude", "longitude", "user_score", "description",
    "price_per_night", "nb_reviews", "stars", "property_type",
    "amenities", "distance_center_km", "image_url",
]

hotels_df = pd.DataFrame(all_hotels)
for col in HOTEL_COLUMNS:
    if col not in hotels_df.columns:
        hotels_df[col] = None
hotels_df = hotels_df[HOTEL_COLUMNS]

hotels_df.to_csv("src/hotels_data.csv", index=False, encoding="utf-8")
print(f"Sauvegardé : src/hotels_data.csv ({len(hotels_df)} hôtels)")
hotels_df.head()


Sauvegardé : src/hotels_data.csv (700 hôtels)


,city_id,city,hotel_name,url,latitude,longitude,user_score,description,price_per_night,nb_reviews,stars,property_type,amenities,distance_center_km,image_url
0,1,Mont Saint Michel,La Vieille Auberge,https://www.booking.com/hotel/fr/la-vieille-auberge-le-mont-saint-michel.fr....,48.636063,-1.511457,7.6,La Vieille Auberge vous accueille dans le village médiéval du Mont-Saint-Mic...,127,None,4.0,NaN,", Chambres non-fumeurs, Chambres non-fumeurs, , Restaurant, Restaurant, , Co...",NaN,https://cf.bstatic.com/xdata/images/hotel/square240/47768676.webp?k=ee957b23...
1,1,Mont Saint Michel,Hôtel Vert,https://www.booking.com/hotel/fr/vert.fr.html?aid=304142&label=gen173nr-10CA...,48.614700,-1.509617,8.2,"L’Hotel Vert vous propose des chambres décorées dans des tons pastel, dotées...",80,None,4.0,NaN,", Chambres non-fumeurs, Chambres non-fumeurs, , 2 restaurants, 2 restaurants...",2.4,https://cf.bstatic.com/xdata/images/hotel/square240/122592840.webp?k=57077bc...
2,1,Mont Saint Michel,Auberge Saint Pierre,https://www.booking.com/hotel/fr/auberge-saint-pierre.fr.html?aid=304142&lab...,48.635688,-1.509883,8.0,"Située sur le Mont-Saint-Michel, l'Auberge Saint Pierre est composée de peti...",184,None,6.0,NaN,", Chambres non-fumeurs, Chambres non-fumeurs, , Restaurant, Restaurant, , Co...",NaN,https://cf.bstatic.com/xdata/images/hotel/square240/4318852.webp?k=61fc83dd8...
3,1,Mont Saint Michel,La Mère Poulard,https://www.booking.com/hotel/fr/la-mere-poulard.fr.html?aid=304142&label=ge...,48.635058,-1.510944,7.7,Séjournez dans un hôtel historique au cœur du Mont Saint-Michel.\n\nL'Hôtel ...,277,None,6.0,Petit-déjeuner compris,", Chambres non-fumeurs, Chambres non-fumeurs, , Restaurant, Restaurant, , Co...",NaN,https://cf.bstatic.com/xdata/images/hotel/square240/646102039.webp?k=4785443...
4,1,Mont Saint Michel,Le Saint Aubert,https://www.booking.com/hotel/fr/hotel-saint-aubert.fr.html?aid=304142&label...,48.612938,-1.510105,7.5,"Set amid lush greenery, just 2 km from the Mont Saint-Michel, the hotel welc...",93,None,6.0,NaN,", Chambres non-fumeurs, Chambres non-fumeurs, , Équipements pour les personn...",2.6,https://cf.bstatic.com/xdata/images/hotel/square240/98510837.webp?k=c41ee9fa...


### CSV enrichi (livrable « données météo + hôtels »)

Jointure entre **hôtels** et **scores météo par ville** pour un fichier unique `enriched_kayak.csv`, déposé ensuite sur S3 avec les autres exports.


In [17]:
score_cols = ["city_id", "city", "avg_score", "avg_temp", "total_rain", "avg_humidity", "rank"]
enriched_kayak = hotels_df.merge(
    city_scores[[c for c in score_cols if c in city_scores.columns]],
    on=["city_id", "city"],
    how="left",
)
enriched_kayak.to_csv("src/enriched_kayak.csv", index=False, encoding="utf-8")
print(f"Sauvegardé : src/enriched_kayak.csv ({len(enriched_kayak)} lignes)")
enriched_kayak.head()


Sauvegardé : src/enriched_kayak.csv (700 lignes)


,city_id,city,hotel_name,url,latitude,longitude,user_score,description,price_per_night,nb_reviews,stars,property_type,amenities,distance_center_km,image_url,avg_score,avg_temp,total_rain,avg_humidity,rank
0,1,Mont Saint Michel,La Vieille Auberge,https://www.booking.com/hotel/fr/la-vieille-auberge-le-mont-saint-michel.fr....,48.636063,-1.511457,7.6,La Vieille Auberge vous accueille dans le village médiéval du Mont-Saint-Mic...,127,None,4.0,NaN,", Chambres non-fumeurs, Chambres non-fumeurs, , Restaurant, Restaurant, , Co...",NaN,https://cf.bstatic.com/xdata/images/hotel/square240/47768676.webp?k=ee957b23...,46.2,10.92,7.09,66.75,25
1,1,Mont Saint Michel,Hôtel Vert,https://www.booking.com/hotel/fr/vert.fr.html?aid=304142&label=gen173nr-10CA...,48.614700,-1.509617,8.2,"L’Hotel Vert vous propose des chambres décorées dans des tons pastel, dotées...",80,None,4.0,NaN,", Chambres non-fumeurs, Chambres non-fumeurs, , 2 restaurants, 2 restaurants...",2.4,https://cf.bstatic.com/xdata/images/hotel/square240/122592840.webp?k=57077bc...,46.2,10.92,7.09,66.75,25
2,1,Mont Saint Michel,Auberge Saint Pierre,https://www.booking.com/hotel/fr/auberge-saint-pierre.fr.html?aid=304142&lab...,48.635688,-1.509883,8.0,"Située sur le Mont-Saint-Michel, l'Auberge Saint Pierre est composée de peti...",184,None,6.0,NaN,", Chambres non-fumeurs, Chambres non-fumeurs, , Restaurant, Restaurant, , Co...",NaN,https://cf.bstatic.com/xdata/images/hotel/square240/4318852.webp?k=61fc83dd8...,46.2,10.92,7.09,66.75,25
3,1,Mont Saint Michel,La Mère Poulard,https://www.booking.com/hotel/fr/la-mere-poulard.fr.html?aid=304142&label=ge...,48.635058,-1.510944,7.7,Séjournez dans un hôtel historique au cœur du Mont Saint-Michel.\n\nL'Hôtel ...,277,None,6.0,Petit-déjeuner compris,", Chambres non-fumeurs, Chambres non-fumeurs, , Restaurant, Restaurant, , Co...",NaN,https://cf.bstatic.com/xdata/images/hotel/square240/646102039.webp?k=4785443...,46.2,10.92,7.09,66.75,25
4,1,Mont Saint Michel,Le Saint Aubert,https://www.booking.com/hotel/fr/hotel-saint-aubert.fr.html?aid=304142&label...,48.612938,-1.510105,7.5,"Set amid lush greenery, just 2 km from the Mont Saint-Michel, the hotel welc...",93,None,6.0,NaN,", Chambres non-fumeurs, Chambres non-fumeurs, , Équipements pour les personn...",2.6,https://cf.bstatic.com/xdata/images/hotel/square240/98510837.webp?k=c41ee9fa...,46.2,10.92,7.09,66.75,25


---
## 6. Data Lake — upload vers S3

Préfixe obligatoire du lab : `Bloc-1/Kayak/raw/`.


In [18]:
import boto3
from botocore.exceptions import ClientError

session = boto3.Session(
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)
s3_client = session.client("s3")
s3_resource = session.resource("s3")

try:
    s3_client.create_bucket(
        Bucket=S3_BUCKET_NAME,
        CreateBucketConfiguration={"LocationConstraint": AWS_REGION},
    )
    print(f"Bucket créé : {S3_BUCKET_NAME}")
except ClientError as e:
    err = e.response["Error"]["Code"]
    if err in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
        print(f"Bucket existant : {S3_BUCKET_NAME}")
    else:
        raise

bucket = s3_resource.Bucket(S3_BUCKET_NAME)


Bucket existant : cloudlab-certification-b6bff020


In [19]:
def s3_key(name):
    """Clé S3 sous le préfixe du projet (ex. Bloc-1/Kayak/raw/cities.csv)."""
    return f"{S3_PREFIX}/raw/{name}"


cities_df.to_csv("src/cities.csv", index=False, encoding="utf-8")

files_to_upload = {
    "src/cities.csv":              s3_key("cities.csv"),
    "src/weather_data.csv":        s3_key("weather_data.csv"),
    "src/city_weather_scores.csv": s3_key("city_weather_scores.csv"),
    "src/hotels_data.csv":         s3_key("hotels_data.csv"),
    "src/enriched_kayak.csv":      s3_key("enriched_kayak.csv"),
}

for local, key in files_to_upload.items():
    if os.path.exists(local):
        bucket.upload_file(local, key)
        print(f"  Uploadé : {local} → s3://{S3_BUCKET_NAME}/{key}")
    else:
        print(f"  [!] Fichier manquant : {local}")

print("\nObjets sous le préfixe du projet :")
prefix = f"{S3_PREFIX}/"
for obj in bucket.objects.filter(Prefix=prefix):
    print(f"  {obj.key}  ({obj.size:,} octets)")


  Uploadé : src/cities.csv → s3://cloudlab-certification-b6bff020/Bloc-1/Kayak/raw/cities.csv
  Uploadé : src/weather_data.csv → s3://cloudlab-certification-b6bff020/Bloc-1/Kayak/raw/weather_data.csv
  Uploadé : src/city_weather_scores.csv → s3://cloudlab-certification-b6bff020/Bloc-1/Kayak/raw/city_weather_scores.csv
  Uploadé : src/hotels_data.csv → s3://cloudlab-certification-b6bff020/Bloc-1/Kayak/raw/hotels_data.csv
  Uploadé : src/enriched_kayak.csv → s3://cloudlab-certification-b6bff020/Bloc-1/Kayak/raw/enriched_kayak.csv

Objets sous le préfixe du projet :
  Bloc-1/Kayak/  (0 octets)
  Bloc-1/Kayak/raw/cities.csv  (1,218 octets)
  Bloc-1/Kayak/raw/city_weather_scores.csv  (2,556 octets)
  Bloc-1/Kayak/raw/enriched_kayak.csv  (1,136,626 octets)
  Bloc-1/Kayak/raw/hotels_data.csv  (1,119,218 octets)
  Bloc-1/Kayak/raw/weather_data.csv  (30,428 octets)


---
## 7. ETL : S3 → Neon (PostgreSQL)

**Extract** : lecture des CSV depuis S3. **Transform** : typage, doublons. **Load** : `pandas.to_sql` via SQLAlchemy.

La chaîne `NEON_DATABASE_URL` doit être définie dans `.env` (assertion ci-dessous).


In [20]:
assert NEON_DATABASE_URL, "NEON_DATABASE_URL manquante dans .env — requis pour le Load."


def read_csv_from_s3(bkt, key):
    obj = bkt.Object(key).get()
    return pd.read_csv(StringIO(obj["Body"].read().decode("utf-8")))


cities_raw = read_csv_from_s3(bucket, s3_key("cities.csv"))
weather_raw = read_csv_from_s3(bucket, s3_key("weather_data.csv"))
hotels_raw = read_csv_from_s3(bucket, s3_key("hotels_data.csv"))

print("Extract terminé :")
print(f"  cities  : {cities_raw.shape}")
print(f"  weather : {weather_raw.shape}")
print(f"  hotels  : {hotels_raw.shape}")


Extract terminé :
  cities  : (35, 4)
  weather : (280, 16)
  hotels  : (700, 15)


In [21]:
cities_clean = cities_raw.dropna(subset=["latitude", "longitude"]).copy()
cities_clean["id"] = cities_clean["id"].astype(int)

weather_clean = weather_raw.drop_duplicates(subset=["city_id", "date"]).copy()
weather_clean["date"] = pd.to_datetime(weather_clean["date"])
weather_clean["city_id"] = weather_clean["city_id"].astype(int)
num_cols_w = ["temp_day", "temp_min", "temp_max", "feels_like",
              "humidity", "wind_speed", "pop", "rain_volume"]
if "weather_score" in weather_clean.columns:
    num_cols_w.append("weather_score")
for c in num_cols_w:
    weather_clean[c] = pd.to_numeric(weather_clean[c], errors="coerce")

hotels_clean = hotels_raw.drop_duplicates(subset=["hotel_name", "city_id"]).copy()
hotels_clean["city_id"] = hotels_clean["city_id"].astype(int)
for c in ["user_score", "price_per_night", "nb_reviews", "stars",
          "distance_center_km", "latitude", "longitude"]:
    hotels_clean[c] = pd.to_numeric(hotels_clean[c], errors="coerce")

print("Transform terminé :")
print(f"  cities  : {cities_clean.shape}")
print(f"  weather : {weather_clean.shape}")
print(f"  hotels  : {hotels_clean.shape}")


Transform terminé :
  cities  : (35, 4)
  weather : (280, 16)
  hotels  : (700, 15)


In [22]:
from sqlalchemy import create_engine, text as sql_text

raw_url = NEON_DATABASE_URL.strip()
if raw_url.startswith("postgresql://"):
    sqlalchemy_url = raw_url.replace("postgresql://", "postgresql+psycopg2://", 1)
elif raw_url.startswith("postgres://"):
    sqlalchemy_url = raw_url.replace("postgres://", "postgresql+psycopg2://", 1)
else:
    sqlalchemy_url = raw_url

engine = create_engine(sqlalchemy_url)

with engine.connect() as conn:
    v = conn.execute(sql_text("SELECT version()")).fetchone()[0]
    print(f"Connecté à : {v[:70]}…")

cities_clean.to_sql("cities", engine, if_exists="replace", index=False)
print(f"  Table 'cities' créée ({len(cities_clean)} lignes)")

weather_clean.to_sql("weather", engine, if_exists="replace", index=False)
print(f"  Table 'weather' créée ({len(weather_clean)} lignes)")

hotels_clean.to_sql("hotels", engine, if_exists="replace", index=False)
print(f"  Table 'hotels' créée ({len(hotels_clean)} lignes)")

print("\nLoad terminé — données disponibles dans Neon.")


Connecté à : PostgreSQL 17.8 (a284a84) on aarch64-unknown-linux-gnu, compiled by gc…
  Table 'cities' créée (35 lignes)
  Table 'weather' créée (280 lignes)
  Table 'hotels' créée (700 lignes)

Load terminé — données disponibles dans Neon.


In [23]:
with engine.connect() as conn:
    print("Tables disponibles :")
    rows = conn.execute(sql_text(
        "SELECT table_name FROM information_schema.tables "
        "WHERE table_schema = 'public'"
    )).fetchall()
    for r in rows:
        print(f"  - {r[0]}")

    print("\nTop 5 villes par score météo (SQL) :")
    rows = conn.execute(sql_text(
        "SELECT city, ROUND(AVG(weather_score)::numeric, 2) AS avg_score "
        "FROM weather GROUP BY city ORDER BY avg_score DESC LIMIT 5"
    )).fetchall()
    for r in rows:
        print(f"  {r[0]:30s} → {r[1]}")

    print("\nNombre d'hôtels par ville (extrait) :")
    rows = conn.execute(sql_text(
        "SELECT city, COUNT(*) AS nb, ROUND(AVG(user_score)::numeric, 1) AS avg "
        "FROM hotels GROUP BY city ORDER BY nb DESC LIMIT 15"
    )).fetchall()
    for r in rows:
        print(f"  {r[0]:30s} → {r[1]} hôtels (score moy : {r[2]})")


Tables disponibles :
  - cities
  - weather
  - hotels

Top 5 villes par score météo (SQL) :
  Bormes les Mimosas             → 74.61
  Aix en Provence                → 74.28
  Marseille                      → 74.09
  Aigues Mortes                  → 73.79
  Collioure                      → 73.31

Nombre d'hôtels par ville (extrait) :
  Dijon                          → 20 hôtels (score moy : 8.2)
  Lyon                           → 20 hôtels (score moy : 7.9)
  Collioure                      → 20 hôtels (score moy : 8.5)
  Biarritz                       → 20 hôtels (score moy : 8.2)
  Amiens                         → 20 hôtels (score moy : 8.4)
  Besancon                       → 20 hôtels (score moy : 8.2)
  Ariege                         → 20 hôtels (score moy : 8.8)
  Saintes Maries de la mer       → 20 hôtels (score moy : 8.4)
  Marseille                      → 20 hôtels (score moy : 8.0)
  Strasbourg                     → 20 hôtels (score moy : 8.3)
  Toulouse                       

---
## 8. Visualisations finales


In [24]:
fig_dest = px.scatter_mapbox(
    top_5, lat="latitude", lon="longitude",
    size="avg_score", color="avg_temp",
    hover_name="city",
    hover_data={"rank": True, "avg_score": ":.1f", "avg_temp": ":.1f",
                "avg_humidity": ":.0f", "total_rain": ":.1f",
                "latitude": False, "longitude": False},
    color_continuous_scale="Turbo", size_max=35, zoom=5,
    center={"lat": 46.5, "lon": 2.5},
    title="Top 5 des meilleures destinations météo en France",
    labels={"rank": "Classement", "avg_score": "Score météo (/100)",
            "avg_temp": "Temp. moy. (°C)", "avg_humidity": "Humidité (%)",
            "total_rain": "Pluie cumulée (mm)"},
)
fig_dest.add_trace(go.Scattermapbox(
    lat=top_5["latitude"], lon=top_5["longitude"], mode="text",
    text=top_5.apply(lambda r: f"#{int(r['rank'])} {r['city']}", axis=1),
    textposition="top center",
    textfont=dict(size=12, color="black"), showlegend=False,
))
fig_dest.update_layout(mapbox_style="open-street-map",
                       margin=dict(r=0, t=50, l=0, b=0), height=650)
fig_dest.show()


In [25]:
top_20_hotels = (
    hotels_df
    .dropna(subset=["user_score", "latitude", "longitude"])
    .sort_values("user_score", ascending=False)
    .head(20).copy()
)

color_col = "price_per_night" if top_20_hotels["price_per_night"].notna().any() else "user_score"

fig_htl = px.scatter_mapbox(
    top_20_hotels, lat="latitude", lon="longitude",
    size="user_score", color=color_col,
    hover_name="hotel_name",
    hover_data={"city": True, "user_score": ":.1f", "price_per_night": True,
                "stars": True, "nb_reviews": True,
                "distance_center_km": ":.1f",
                "latitude": False, "longitude": False},
    color_continuous_scale="Viridis_r", size_max=25, zoom=5,
    center={"lat": 46.5, "lon": 2.5},
    title="Top 20 des meilleurs hôtels",
    labels={"city": "Ville", "user_score": "Score", "price_per_night": "Prix/nuit (€)",
            "stars": "Étoiles", "nb_reviews": "Avis", "distance_center_km": "Dist. centre (km)"},
)
fig_htl.update_layout(mapbox_style="open-street-map",
                      margin=dict(r=0, t=50, l=0, b=0), height=650)
fig_htl.update_traces(marker=dict(opacity=0.9))
fig_htl.show()


In [26]:
fig_bar = px.bar(
    top_5.sort_values("avg_score"), x="avg_score", y="city",
    orientation="h", color="avg_temp", color_continuous_scale="RdYlBu_r",
    title="Comparaison météo des Top-5 destinations",
    labels={"avg_score": "Score météo moyen", "city": "", "avg_temp": "Temp. moy. (°C)"},
)
fig_bar.update_layout(height=400, yaxis=dict(autorange="reversed"))
fig_bar.show()

valid_qp = hotels_df.dropna(subset=["user_score", "price_per_night"])
if not valid_qp.empty:
    has_reviews = valid_qp["nb_reviews"].notna().any()
    fig_qp = px.scatter(
        valid_qp, x="price_per_night", y="user_score",
        color="city",
        size="nb_reviews" if has_reviews else None,
        hover_name="hotel_name",
        title="Rapport qualité / prix des hôtels",
        labels={"price_per_night": "Prix par nuit (€)", "user_score": "Score",
                "city": "Ville", "nb_reviews": "Avis"},
    )
    fig_qp.update_layout(height=500)
    fig_qp.show()


---
## 9. Conformité RGPD

Le **RGPD** encadre les données personnelles. Ici : **géolocalisation de villes**, **météo**, **données publiques d'hébergements** — pas de données directement identifiables sur des personnes physiques dans le périmètre choisi.

**Mesures** : respect des limites d'usage (Nominatim 1 req/s, délais scraping), **minimisation**, credentials dans `.env` (non versionné), accès **AWS IAM** limité au bucket S3, connexion **Neon en TLS** (chaîne avec `sslmode=require`). Les utilisateurs peuvent supprimer les données des buckets / tables pour exercer leur contrôle sur les jeux exportés.


---
## 10. Conclusion

- **35 villes** géolocalisées et couvertes météo sur 7 jours.
- **Hôtels** collectés pour chaque ville (selon disponibilité Booking).
- **S3** : zone `Bloc-1/Kayak/raw/` avec CSV normalisés + `enriched_kayak.csv`.
- **Neon** : tables `cities`, `weather`, `hotels` pour l'analytique SQL.
- **Visualisations** : Top-5 destinations, Top-20 hôtels, graphiques complémentaires.

**Pistes** : automatisation planifiée du pipeline, API hôtelière officielle, monitoring des exécutions.
